In [ ]:
"""
XGBoost Model for Predicting Flexural Strength of Gypsum Composites
Author: Haseeb Ahmad
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import xgboost as xgb

warnings.filterwarnings("ignore")


# ==========================================
# 1. Load and Prepare Data
# ==========================================

def load_data(file_path):

    df = pd.read_csv(file_path, header=2)
    df_array = np.asarray(df)

    # Input Features
    X = np.column_stack([
        df_array[0:161, 4].astype(float),
        df_array[0:161, 5].astype(float),
        df_array[0:161, 6].astype(float),
        df_array[0:161, 7].astype(float),
        df_array[0:161, 8].astype(float),
        df_array[0:161, 9].astype(float),
        df_array[0:161, 10].astype(float)
    ])

    # Target (Flexural Strength)
    y = df_array[0:161, 12].astype(float)

    # Remove NaN rows
    mask = ~np.isnan(y)
    return X[mask], y[mask]


# ==========================================
# 2. Train Model
# ==========================================

def train_model(X_train, y_train):

    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse"
    )

    model.fit(X_train, y_train)
    return model


# ==========================================
# 3. Evaluate Model
# ==========================================

def evaluate_model(model, X_train, X_test, y_train, y_test):

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    results = {
        "Train R2": r2_score(y_train, y_train_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
    }

    return results, y_test_pred


# ==========================================
# 4. Convergence Plot
# ==========================================

def plot_convergence(model, X_train, X_test, y_train, y_test, save_path=None):

    train_r2, test_r2 = [], []

    for i in range(1, model.n_estimators + 1):

        y_train_pred = model.predict(X_train, iteration_range=(0, i))
        y_test_pred = model.predict(X_test, iteration_range=(0, i))

        train_r2.append(r2_score(y_train, y_train_pred))
        test_r2.append(r2_score(y_test, y_test_pred))

    plt.figure(figsize=(6, 4))
    plt.plot(train_r2, label="Training R²")
    plt.plot(test_r2, label="Testing R²")
    plt.xlabel("Boosting Rounds")
    plt.ylabel("R² Score")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


# ==========================================
# 5. Actual vs Predicted Plot (±10% & ±20%)
# ==========================================

def plot_actual_vs_predicted(y_true, y_pred, save_path=None):

    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    x_vals = np.linspace(y_true.min(), y_true.max(), 200)

    perfect_fit = x_vals
    err10 = 0.10 * x_vals
    err20 = 0.20 * x_vals

    plt.figure(figsize=(6, 4))
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolors='k', label='Data')
    plt.plot(x_vals, perfect_fit, 'r--', lw=2, label='Perfect Fit')

    plt.fill_between(x_vals,
                     perfect_fit - err20,
                     perfect_fit + err20,
                     alpha=0.15,
                     label='±20% Error')

    plt.fill_between(x_vals,
                     perfect_fit - err10,
                     perfect_fit + err10,
                     alpha=0.25,
                     label='±10% Error')

    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Values')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


# ==========================================
# 6. Feature Importance
# ==========================================

def plot_feature_importance(model, X_test, y_test, feature_names, save_path=None):

    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=20,
        scoring="neg_mean_squared_error",
        random_state=42
    )

    importances = np.abs(result.importances_mean)
    indices = np.argsort(importances)

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(importances)), importances[indices])
    plt.yticks(range(len(importances)), np.array(feature_names)[indices])
    plt.xlabel("Permutation Importance")
    plt.grid(True)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)

    plt.show()


# ==========================================
# 7. Main Execution
# ==========================================

def main():

    data_path = os.path.join("data", "Gypsum_updated.csv")
    os.makedirs("figures", exist_ok=True)

    X, y = load_data(data_path)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Scaling (optional for XGBoost but kept for consistency)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = train_model(X_train_scaled, y_train)

    results, y_pred = evaluate_model(
        model,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )

    print("\nModel Performance:")
    for key, value in results.items():
        print(f"{key}: {value:.4f}")

    feature_names = [
        "Gypsum Strength",
        "Gypsum Quantity",
        "Water Quantity",
        "Water/Gypsum Ratio",
        "Wheat Straw",
        "CaCl2",
        "Ca(OH)2"
    ]

    plot_convergence(
        model,
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
        save_path="figures/convergence.png"
    )

    plot_actual_vs_predicted(
        y_test,
        y_pred,
        save_path="figures/actual_vs_predicted.png"
    )

    plot_feature_importance(
        model,
        X_test_scaled,
        y_test,
        feature_names,
        save_path="figures/feature_importance.png"
    )


if __name__ == "__main__":
    main()